# 7.02 π0 架构精读：动作专家与共享注意力

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROI-su7/vla-ladder/blob/main/notebooks/07_vla/7.02_pi0_architecture.ipynb)


## 为什么要学

π0 最巧妙的设计是**动作专家**（action expert）：在 30 亿参数的 VLM 旁边，挂一个 3 亿参数的小 Transformer 专门处理动作 token。两者**层数相同、每层共享一次注意力**，但权重完全独立。这种结构叫 **Mixture-of-Transformers（MoT）**。

为什么不直接用 VLM 处理动作 token？

- VLM 是在图文数据上预训练的，动作 token 的统计特性和文字完全不同，直接混在一起会破坏 VLM 的知识；
- 动作专家可以更小（宽 1024 vs 2048），推理时 10 步去噪只需要跑小模型，速度快很多。

本课逐段精读 openpi 的 `gemma_pytorch.py` 与 `pi0_pytorch.py`，并在 mini-π0 中亲手实现共享注意力层。

---

## 1. 全景图

```text
前缀 prefix（VLM 权重处理）                       后缀 suffix（动作专家权重处理）
[img × 256 × 3 相机] [文字 token]           π0:  [状态] [a_1 + t] ... [a_50 + t]
          │                                  π0.5:        [a_1] ... [a_50]   （t 通过 adaRMSNorm 注入每层）
          │  每一层：                                   │
          ├── VLM 的 q_proj/k_proj/v_proj ──┐   ┌── 专家的 q_proj/k_proj/v_proj ──┤
          │                                  ▼   ▼                                 │
          │                  拼接 q、k、v（沿序列维）→ RoPE → 一次注意力（块状因果掩码）
          │                                  │   │                                 │
          ├── VLM 的 o_proj、MLP ◀──────────┘   └──────────▶ 专家的 o_proj、MLP ──┤
          ▼                                                                         ▼
     （训练时不用）                                         action_out_proj → 速度 v_t (50, 32)
```

## 2. 共享注意力：`PaliGemmaWithExpertModel.forward`

openpi 中 `compute_layer_complete` 的核心逻辑（已删减 dtype 转换和梯度检查点）：

```py
for i, hidden_states in enumerate(inputs_embeds):          # i=0: 前缀(VLM)，i=1: 后缀(动作专家)
    layer = models[i].layers[layer_idx]
    hidden_states, gate = layer.input_layernorm(hidden_states, cond=adarms_cond[i])
    query_states.append(layer.self_attn.q_proj(hidden_states).view(hidden_shape).transpose(1, 2))
    key_states.append(...k_proj...); value_states.append(...v_proj...)
query_states = torch.cat(query_states, dim=2)              # 沿序列维拼接！
key_states = torch.cat(key_states, dim=2)
value_states = torch.cat(value_states, dim=2)
cos, sin = self.paligemma.model.language_model.rotary_emb(dummy_tensor, position_ids)
query_states, key_states = apply_rotary_pos_emb(query_states, key_states, cos, sin)
att_output, _ = eager_attention_forward(..., query_states, key_states, value_states, attention_mask, scaling)
start_pos = 0
for i, hidden_states in enumerate(inputs_embeds):          # 切回两部分，各自过自己的 o_proj、残差、MLP
    end_pos = start_pos + hidden_states.shape[1]
    out_emb = layer.self_attn.o_proj(att_output[:, start_pos:end_pos])
    out_emb = _gated_residual(hidden_states, out_emb, gates[i])
    ...post_attention_layernorm, mlp, _gated_residual...
```

关键点：

1. 两个专家的 `num_heads × head_dim` 必须相同（都是 8 × 256），否则 q/k/v 无法沿序列维拼接；但**宽度（hidden size）可以不同**，因为 `q_proj` 把各自的宽度映射到同一个 `8×256` 空间，`o_proj` 再映射回各自的宽度。
2. **掩码决定信息流**（4.01 课）：前缀看不到后缀，后缀能看到前缀。所以 VLM 部分的计算与动作无关。
3. 训练时两部分一起算（一次前向）；推理时前缀只算一次并缓存 KV。

### 练习 1：实现一层共享注意力

用 mini-π0 的两个 `ExpertLayer`（`vlm_layer`、`expert_layer`，它们只提供权重：`norm`、`qkv`、`o_proj`、`mlp`），实现一层 MoT 的前向。为了简单，这里只处理 π0（没有 adaRMSNorm，`gate` 都是 None，残差就是普通相加）。

In [ ]:
# 🔧 环境准备（每个 notebook 都先运行这一格）
# - 在 Colab 中：自动下载本项目代码
# - 在本地：自动把项目根目录加入 sys.path（也可以在根目录执行 `pip install -e .`）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/vla-ladder"):
        os.system("git clone -q --depth 1 https://github.com/ROI-su7/vla-ladder.git /content/vla-ladder")
    sys.path.insert(0, "/content/vla-ladder")
else:
    _p = os.path.abspath(".")
    while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "vla_ladder")):
        _p = os.path.dirname(_p)
    sys.path.insert(0, _p)
print("环境就绪 ✅")

In [ ]:
import math
import torch
import torch.nn.functional as F
from vla_ladder.envs import CharTokenizer, ToyTableEnv
from vla_ladder.models.mini_pi0 import MiniPi0, MiniPi0Config
from vla_ladder.models.transformer import apply_rope, rope_cos_sin, make_att_2d_masks
from vla_ladder.utils import set_seed, count_params

def mot_layer(vlm_layer, expert_layer, prefix_h, suffix_h, mask, position_ids, head_dim):
    """
    prefix_h: (B, Lp, Wv)，suffix_h: (B, Ls, We)
    mask: bool (B, Lp+Ls, Lp+Ls)；position_ids: (B, Lp+Ls)
    返回 (新的 prefix_h, 新的 suffix_h)
    """
    # TODO:
    # 1) 各自归一化：hp, _ = vlm_layer.norm("input", prefix_h, None)；hs 同理
    # 2) 各自算 q/k/v：qp, kp, vp = vlm_layer.qkv(hp)；qs, ks, vs = expert_layer.qkv(hs)   每个 (B, h, L, hd)
    # 3) 沿序列维拼接（dim=2），apply_rope(q, k, *rope_cos_sin(position_ids, head_dim))
    # 4) att = F.scaled_dot_product_attention(q, k, v, attn_mask=mask[:, None])，再 transpose(1,2).reshape(B, L, -1)
    # 5) 切回两部分，各自 o_proj + 残差；再各自 norm("post") + mlp + 残差
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
import math
import torch
import torch.nn.functional as F
from vla_ladder.envs import CharTokenizer, ToyTableEnv
from vla_ladder.models.mini_pi0 import MiniPi0, MiniPi0Config
from vla_ladder.models.transformer import apply_rope, rope_cos_sin, make_att_2d_masks
from vla_ladder.utils import set_seed, count_params

def mot_layer(vlm_layer, expert_layer, prefix_h, suffix_h, mask, position_ids, head_dim):
    hp, _ = vlm_layer.norm("input", prefix_h, None)
    hs, _ = expert_layer.norm("input", suffix_h, None)
    qp, kp, vp = vlm_layer.qkv(hp)
    qs, ks, vs = expert_layer.qkv(hs)
    q, k, v = torch.cat([qp, qs], 2), torch.cat([kp, ks], 2), torch.cat([vp, vs], 2)
    q, k = apply_rope(q, k, *rope_cos_sin(position_ids, head_dim))
    att = F.scaled_dot_product_attention(q, k, v, attn_mask=mask[:, None])
    B, _, L, _ = att.shape
    att = att.transpose(1, 2).reshape(B, L, -1)
    Lp = prefix_h.shape[1]
    outs = []
    for layer, h_in, a in [(vlm_layer, prefix_h, att[:, :Lp]), (expert_layer, suffix_h, att[:, Lp:])]:
        x = h_in + layer.o_proj(a)
        h, _ = layer.norm("post", x, None)
        outs.append(x + layer.mlp(h))
    return outs[0], outs[1]

In [ ]:
# ✅ 自测：与 mini-π0 的 MixtureOfTransformers（只取 1 层）结果一致
set_seed(0)
tok = CharTokenizer()
cfg = MiniPi0Config(vocab_size=tok.vocab_size, depth=1)
model = MiniPi0(cfg).eval()
env = ToyTableEnv(num_envs=3, seed=0); obs = env.reset()
ids, msk = tok.encode(obs["instruction"])
images, lang, lang_mask = torch.tensor(obs["image"]), torch.tensor(ids), torch.tensor(msk)
state = torch.tensor(obs["state"])
x_t, t = torch.randn(3, cfg.action_horizon, 2), torch.rand(3)
with torch.no_grad():
    pe, pp, pa = model.embed_prefix(images, lang, lang_mask)
    se, sp, sa, _ = model.embed_suffix(state, x_t, t)
    pad, att = torch.cat([pp, sp], 1), torch.cat([pa, sa], 1)
    mask, pos = make_att_2d_masks(pad, att), torch.cumsum(pad.long(), 1) - 1
    my_p, my_s = mot_layer(model.moe.vlm_layers[0], model.moe.expert_layers[0], pe, se, mask, pos, cfg.head_dim)
    # 参考实现最后还有一个 final norm，这里手动加上再比较
    (ref_p, ref_s), _ = model.moe([pe, se], mask, pos)
assert torch.allclose(model.moe.vlm_norm(my_p)[pp], ref_p[pp], atol=1e-5)
assert torch.allclose(model.moe.expert_norm(my_s), ref_s, atol=1e-5)
print("通过 ✅ 你实现了 π0 的核心：两套权重、一次注意力")

---

## 3. 后缀：`embed_suffix`（π0 vs π0.5）

### π0

```py
state_emb = self.state_proj(state)                         # 状态 -> 1 个 token
att_masks += [1]                                           # 新块：前缀看不到状态
time_emb = create_sinusoidal_pos_embedding(timestep, width, min_period=4e-3, max_period=4.0)
action_emb = self.action_in_proj(noisy_actions)            # (B, 50, 1024)
action_time_emb = cat([action_emb, time_emb.expand_as(action_emb)], dim=2)
action_time_emb = self.action_time_mlp_out(F.silu(self.action_time_mlp_in(action_time_emb)))
att_masks += [1] + [0] * (action_horizon - 1)              # 新块：动作之间双向
```

时间 $t$ 和每个带噪动作**拼接后过一个 MLP**，作为动作 token 的输入。

### π0.5

```py
time_emb = self.time_mlp_out(F.silu(self.time_mlp_in(time_emb)))
adarms_cond = F.silu(time_emb)                             # 时间 -> 一个条件向量
action_time_emb = action_emb                               # 动作 token 里不再有时间
# 动作专家的每个 RMSNorm 变成 adaRMSNorm(x, cond=adarms_cond)：scale/shift/gate 都由 t 决定
```

π0.5 让时间**调制动作专家的每一层**（和 Diffusion Policy 的 FiLM、DiT 的 adaLN 同一思路），条件信号更强。状态则离散化后放进了前缀的文字里。

### π0.5 的另一个关键：知识隔离（Knowledge Insulation）

π0.5 训练时，VLM 同时用 **FAST 离散动作 token** 做下一个 token 预测（交叉熵），动作专家用流匹配学连续动作，并且**阻断动作专家到 VLM 的梯度**（stop-gradient）。目的：防止动作专家的梯度破坏 VLM 的语义知识，同时让 VLM 自己也学会"理解动作"。这也是 openpi 中 π0.5 基础模型泛化好的原因之一。

### 练习 2：数一数 mini-π0 的参数都在哪

In [ ]:
def param_breakdown(model):
    """返回 dict：视觉塔、词嵌入、VLM 层、动作专家层、其它（投影层/时间 MLP 等）各自的参数量"""
    # TODO: 用 count_params 分别统计 model.vision、model.embed_tokens、model.moe.vlm_layers、model.moe.expert_layers，
    #       其余 = 总数 - 以上之和
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def param_breakdown(model):
    """返回 dict：视觉塔、词嵌入、VLM 层、动作专家层、其它（投影层/时间 MLP 等）各自的参数量"""
    parts = {
        "vision": count_params(model.vision),
        "embed_tokens": count_params(model.embed_tokens),
        "vlm_layers": count_params(model.moe.vlm_layers),
        "expert_layers": count_params(model.moe.expert_layers),
    }
    parts["other"] = count_params(model) - sum(parts.values())
    return parts

In [ ]:
# ✅ 自测
for pi05 in [False, True]:
    m = MiniPi0(MiniPi0Config(vocab_size=tok.vocab_size, pi05=pi05))
    b = param_breakdown(m)
    assert sum(b.values()) == count_params(m)
    print(("π0.5" if pi05 else "π0  "), {k: f"{v / 1e3:.0f}K" for k, v in b.items()})
print("通过 ✅（π0.5：动作专家多了 adaRMSNorm 的 dense 层，other 里有离散状态 embedding 与 time_mlp；π0：other 里是 state_proj 与 action_time_mlp）")

---

## 4. 推理：`sample_actions` 与 `denoise_step`

```py
# 1) 前缀只算一次，得到每层的 KV cache
_, past_key_values = self.paligemma_with_expert.forward(inputs_embeds=[prefix_embs, None], use_cache=True, ...)
# 2) 10 步 Euler：每步只跑动作专家
while time >= -dt / 2:
    v_t = self.denoise_step(state, prefix_pad_masks, past_key_values, x_t, expanded_time)
    x_t = x_t + dt * v_t
```

`denoise_step` 里的掩码构造：后缀的 query 要能看到 **[全部有效前缀 + 后缀自己的块状掩码]**：

```py
prefix_pad_2d_masks = prefix_pad_masks[:, None, :].expand(batch_size, suffix_len, prefix_len)
suffix_att_2d_masks = make_att_2d_masks(suffix_pad_masks, suffix_att_masks)
full_att_2d_masks = torch.cat([prefix_pad_2d_masks, suffix_att_2d_masks], dim=2)   # (B, Ls, Lp + Ls)
position_ids = prefix_offsets + torch.cumsum(suffix_pad_masks, dim=1) - 1          # 位置号接在前缀后面
```

### 练习 3：验证"KV cache 推理 == 完整前向"

固定噪声和时间，比较两种方式算出的速度 $v_t$：

- 方式 A：`model.forward(...)` 返回的是逐元素损失 $(u_t-v_t)^2$，可以反推出 $v_t$ 吗？不方便。改为直接复现 forward 里的计算：拼接前缀和后缀、一次过 `model.moe`、取最后 H 个 token、`action_out_proj`；
- 方式 B：`prefix_cache` + `denoise_step`。

In [ ]:
@torch.no_grad()
def velocity_full(model, images, lang, lang_mask, state, x_t, t):
    # TODO: 仿照 MiniPi0.forward（但不计算损失），返回 v_t (B, H, A)
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
@torch.no_grad()
def velocity_full(model, images, lang, lang_mask, state, x_t, t):
    pe, pp, pa = model.embed_prefix(images, lang, lang_mask, state)
    se, sp, sa, cond = model.embed_suffix(state, x_t, t)
    pad, att = torch.cat([pp, sp], 1), torch.cat([pa, sa], 1)
    (_, out), _ = model.moe([pe, se], make_att_2d_masks(pad, att), torch.cumsum(pad.long(), 1) - 1, adarms_cond=cond)
    return model.action_out_proj(out[:, -model.cfg.action_horizon :])

In [ ]:
# ✅ 自测（π0 与 π0.5 都要一致）
for pi05 in [False, True]:
    set_seed(0)
    m = MiniPi0(MiniPi0Config(vocab_size=tok.vocab_size, pi05=pi05)).eval()
    vA = velocity_full(m, images, lang, lang_mask, state, x_t, t)
    with torch.no_grad():
        cache, ppad, _ = m.prefix_cache(images, lang, lang_mask, state)
        vB = m.denoise_step(state, ppad, cache, x_t, t)
    assert torch.allclose(vA, vB, atol=1e-5), (vA - vB).abs().max()
print("通过 ✅ KV cache 推理与完整前向完全一致")

再看看 KV cache 带来的加速（10 步去噪，前缀只算 1 次 vs 每步都算）：

In [ ]:
import time as _time
m = MiniPi0(MiniPi0Config(vocab_size=tok.vocab_size)).eval()
B = 64
env = ToyTableEnv(num_envs=B, seed=1); o = env.reset(); ids, msk = tok.encode(o["instruction"])
I, Lg, Lm, S = torch.tensor(o["image"]), torch.tensor(ids), torch.tensor(msk), torch.tensor(o["state"])
with torch.no_grad():
    t0 = _time.time()
    for _ in range(3):
        m.sample_actions(I, Lg, Lm, S)
    t_cache = (_time.time() - t0) / 3
    t0 = _time.time()
    for _ in range(3):
        x = torch.randn(B, m.cfg.action_horizon, 2)
        for k in range(10):
            x = x - 0.1 * velocity_full(m, I, Lg, Lm, S, x, torch.full((B,), 1 - 0.1 * k))
    t_full = (_time.time() - t0) / 3
print(f"有 KV cache：{t_cache * 1000:.0f} ms；每步重算前缀：{t_full * 1000:.0f} ms；加速 {t_full / t_cache:.1f}×")

在 π0 上，前缀有 800 个左右的 token、VLM 有 2B 参数，而动作专家只有 300M、后缀 51 个 token，KV cache 的加速比远大于这里的玩具模型。

---

## 小结

| 组件 | π0 | π0.5 |
|---|---|---|
| 前缀 | 图像 + 文字（双向） | 图像 + 文字 + **离散状态**（双向） |
| 后缀 | 状态 token + 动作 token（时间拼接 + MLP） | 动作 token（时间通过 **adaRMSNorm** 注入每层） |
| 注意力 | VLM 与动作专家每层拼接 q/k/v，一次注意力 | 同左 |
| 训练 | 流匹配 | 流匹配 + FAST token 交叉熵 + 知识隔离 |
| 推理 | 前缀 KV cache + 10 步 Euler | 同左 |

下一课：把这个模型训练起来。